In [ ]:
use mavenmovies;
/*
Question 1
Display the names of customers who have rented more movies than the
average number of rentals made by all customers.
*/

with customer_rentals as (
select c.customer_id, concat(c.first_name, ' ', c.last_name) as customer_name, count(r.rental_id) as rental_count
from customer c
join rental r on c.customer_id = r.customer_id
group by c.customer_id, c.first_name, c.last_name
)
select customer_name, rental_count
from customer_rentals
where rental_count > (
select avg(rental_count) from customer_rentals
);

/*
Question 2
Display each film along with its rental rate and assign a rank based on rental
rate from highest to lowest.
*/

select title, rental_rate, rank() over (order by rental_rate desc) as rate_rank
from film;

/*
Question 3
Create a view that displays customer names along with their email and active status.
*/

create view customer_email_status_view as
select concat(first_name, ' ', last_name) as customer_name, email, active
from customer;

/*
Question 4
Find the top 10 customers who generated the highest payment amount.
*/

select c.customer_id, concat(c.first_name, ' ', c.last_name) as customer_name, sum(p.amount) as total_payment
from customer c
join payment p on c.customer_id = p.customer_id
group by c.customer_id, c.first_name, c.last_name
order by total_payment desc
limit 10;

/*
Question 5
Identify the customers whose total spending on rentals falls within the top 20% of all customers.
*/

with customer_spending as (
select c.customer_id, concat(c.first_name, ' ', c.last_name) as customer_name, sum(p.amount) as total_spent,
ntile(5) over (order by sum(p.amount) desc) as spending_bucket
from customer c
join payment p on c.customer_id = p.customer_id
group by c.customer_id, c.first_name, c.last_name
)
select customer_id, customer_name, total_spent
from customer_spending
where spending_bucket = 1;

/*
Question 6
Display every actor along with the number of movies they acted in and assign
Dense Rank based on movie count.
*/

select a.actor_id, concat(a.first_name, ' ', a.last_name) as actor_name, count(fa.film_id) as movie_count,
dense_rank() over (order by count(fa.film_id) desc) as movie_count_rank
from actor a
left join film_actor fa on a.actor_customer_email_status_viewcustomer_email_status_viewcustomer_nameid = fa.actor_id
group by a.actor_id, a.first_name, a.last_name;

/*
Question 7
Create a view showing movie title, category, rental rate and replacement cost.
*/

create view film_details_view as
select f.title, c.name as category, f.rental_rate, f.replacement_cost
from film f
join film_category fc on f.film_id = fc.film_id
join category c on fc.category_id = c.category_id;

/*
Question 8
Create a stored procedure that returns the top 20 most rented movies.
*/

delimiter //

create procedure gettop20rentedmovies()
begin
select f.film_id, f.title, count(r.rental_id) as rental_count
from film f
join inventory i on f.film_id = i.film_id
join rental r on i.inventory_id = r.inventory_id
group by f.film_id, f.title
order by rental_count desc
limit 20;
end //

delimiter ;

/*
Question 9
Create a procedure that accepts a movie rating and returns all movies of that rating.
*/

delimiter //

create procedure getmoviesbyrating(in p_rating varchar(10))
begin
select film_id, title, rating, rental_rate, replacement_cost
from film
where rating = p_rating;
end //

delimiter ;

/*
Question 10
Identify the top 3 films in each category based on their rental counts.
*/

with film_category_rentals as (
select c.name as category_name, f.title as film_title, count(r.rental_id) as rental_count,
rank() over (partition by c.name order by count(r.rental_id) desc) as rank_in_category
from category c
join film_category fc on c.category_id = fc.category_id
join film f on fc.film_id = f.film_id
join inventory i on f.film_id = i.film_id
join rental r on i.inventory_id = r.inventory_id
group by c.name, f.title
)
select category_name, film_title, rental_count
from film_category_rentals
where rank_in_category <= 3;

/*
Question 11
Calculate the running total of rentals per category, ordered by rental count.
*/

with category_rental_counts as (
select c.name as category_name, count(r.rental_id) as rental_count
from category c
join film_category fc on c.category_id = fc.category_id
join film f on fc.film_id = f.film_id
join inventory i on f.film_id = i.film_id
join rental r on i.inventory_id = r.inventory_id
group by c.name
)
select category_name, rental_count,
sum(rental_count) over (order by rental_count desc, category_name) as running_total
from category_rental_counts;

/*
Question 12
Create a CTE to generate a report showing pairs of actors who have
appeared in the same film together, using the film_actor table
*/

with actor_pairs as (
select fa1.actor_id as actor1_id, fa2.actor_id as actor2_id, fa1.film_id
from film_actor fa1
join film_actor fa2 on fa1.film_id = fa2.film_id and fa1.actor_id < fa2.actor_id
)
select distinct
concat(a1.first_name, ' ', a1.last_name) as actor1_name,
concat(a2.first_name, ' ', a2.last_name) as actor2_name,
ap.film_id,
f.title as film_title
from actor_pairs ap
join actor a1 on ap.actor1_id = a1.actor_id
join actor a2 on ap.actor2_id = a2.actor_id
join film f on ap.film_id = f.film_id;

/*
Question 13
Create a procedure that accepts a customer ID as input and returns
the customer's total payment as an output parameter.
*/

delimiter //

create procedure getcustomertotalpayment(
in p_customer_id int,
out p_total_payment decimal(10,2)
)
begin
select sum(amount) into p_total_payment
from payment
where customer_id = p_customer_id;
end //

delimiter ;